In [1]:
pip install pyserial pymongo


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import time
import random
from datetime import datetime, timezone
from pymongo import MongoClient

# ✅ MongoDB connection settings
MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "sensor_data"
COLLECTION_NAME = "dht11_readings"

# ✅ Sensor metadata
SENSOR_ID = "dht11_simulated"
LOCATION = "Living Room"

# ✅ How many to insert? (0 = infinite loop)
INSERT_COUNT = 20
INTERVAL_SECONDS = 2


def generate_fake_dht11(previous_temp, previous_hum):
    """Generate a realistic fake DHT11 reading."""
    temp = previous_temp + random.uniform(-0.3, 0.3)
    hum = previous_hum + random.uniform(-1.0, 1.0)

    temp = max(0.0, min(50.0, temp))
    hum = max(0.0, min(100.0, hum))

    return round(temp, 1), round(hum, 1)


def main():
        client = MongoClient(MONGO_URI)
        collection = client[DB_NAME][COLLECTION_NAME]
        print(f"✅ Connected to MongoDB: {DB_NAME}.{COLLECTION_NAME}")

        prev_temp, prev_hum = 25.0, 50.0  # Starting baseline
        inserted = 0

        while INSERT_COUNT == 0 or inserted < INSERT_COUNT:
            temp, hum = generate_fake_dht11(prev_temp, prev_hum)
            prev_temp, prev_hum = temp, hum

            doc = {
                "sensor_id": SENSOR_ID,
                "location": LOCATION,
                "temperature_c": temp,
                "humidity_percent": hum,
                "timestamp": datetime.now(timezone.utc),
            }

            result = collection.insert_one(doc)
            inserted += 1
            print(f"[{inserted}] Inserted: {temp}°C, {hum}% (ID: {result.inserted_id})")

            time.sleep(INTERVAL_SECONDS)

        print("✅ Done inserting fake data.")
        client.close()


if __name__ == "__main__":
    main()


✅ Connected to MongoDB: sensor_data.dht11_readings
[1] Inserted: 25.0°C, 49.0% (ID: 68e263ec4d80b651ca2f9041)
[2] Inserted: 25.1°C, 48.8% (ID: 68e263ee4d80b651ca2f9042)
[3] Inserted: 25.1°C, 47.9% (ID: 68e263f04d80b651ca2f9043)
[4] Inserted: 25.2°C, 48.7% (ID: 68e263f24d80b651ca2f9044)
[5] Inserted: 25.1°C, 48.7% (ID: 68e263f44d80b651ca2f9045)
[6] Inserted: 25.2°C, 49.0% (ID: 68e263f64d80b651ca2f9046)


KeyboardInterrupt: 